# Práctica Profesionalizante II · Telco Churn

En Colab, abrí este notebook y ejecutá las celdas en orden. La primera celda te permite cargar el ZIP que acompaña la entrega. En VS Code, abrilo dentro de la carpeta extraída y elegí un kernel Python.

In [ ]:
from pathlib import Path
import sys
import zipfile

root = next((p for p in [Path.cwd(), Path.cwd().parent, Path.cwd() / 'PP2']
             if (p / 'notebooks' / 'etl.py').is_file()), None)
if root is None and 'google.colab' in sys.modules:
    from google.colab import files
    import io
    uploaded = files.upload()
    archive = io.BytesIO(uploaded['Proyecto_PP2.zip'])
    with zipfile.ZipFile(archive) as package:
        package.extractall(Path.cwd())
    root = Path.cwd() / 'PP2'
assert root is not None and (root/'requirements.txt').is_file(), 'Extraé Proyecto_PP2.zip primero.'
sys.path.insert(0, str(root / 'notebooks'))
%pip install -q -r {root / 'requirements.txt'}


## Clase 2 · Exploración y limpieza

Primero inspeccionamos el CSV original. Los valores de cargo acumulado que vinieron como espacios se detectan al limpiar.

In [ ]:
from etl import extraer, limpiar, ejecutar, RESULTS
crudo = extraer()
print(crudo.shape)
crudo.info()
display(crudo.head())
datos, auditoria, resumen = limpiar(crudo)
display(auditoria)
print(resumen)
datos = ejecutar()


El DER está en `data_and_sql/DER_churn.pdf`. En MySQL Workbench, ejecutá `01_crear_churn_db.sql` y luego realizá la carga según el README. Las actividades 3 y 4 están en `consultas_churn.sql` y `clase4_dml_transacciones.sql`.

## Clase 5 · Visualizaciones

In [ ]:
from visualizar import generar, GRAFICOS
from IPython.display import Image, display
generar(datos)
for nombre in ['churn_por_contrato.png', 'densidad_antiguedad.png', 'matriz_decision_financiera.png']:
    display(Image(filename=str(GRAFICOS / nombre), width=700))


## Modelo y decisión financiera

El submuestreo ocurre dentro de los folds de entrenamiento. El umbral se elige en validación y se mide con el conjunto de prueba.

In [ ]:
import pandas as pd
from modelar import modelar
from explorar import analizar
modelo = modelar(datos)
display(pd.read_csv(RESULTS / 'comparacion_modelos.csv'))
display(pd.DataFrame(modelo['politicas']).T)
display(analizar(datos))
